# 🖼️ Миграция фото в Cloudinary — Мебрид

Этот ноутбук:
1. Читает Excel-файл с товарами
2. Скачивает фото по ссылкам из колонки **«Фото»**
3. Загружает их в ваш аккаунт Cloudinary под структурированными именами
4. Сохраняет обновлённый Excel, где колонка «Фото» заменена на Cloudinary-URL

## Что вам понадобится
- Аккаунт на [cloudinary.com](https://cloudinary.com) (бесплатный тариф даёт 25 кредитов/мес, хватит с запасом)
- Credentials со страницы **Dashboard → Product Environment Credentials**: `cloud_name`, `api_key`, `api_secret`
- Excel-файл в колонке «Фото» — ссылки через `\n` (как в исходниках)

## Как запустить
1. Загрузите свой Excel через панель слева (значок 📁 → `Upload`)
2. Впишите credentials и имя файла в ячейке настроек
3. **Runtime → Run all** (или ⌘/Ctrl+F9)
4. Скачайте полученный файл через панель слева

In [ ]:
# --- Установка зависимостей ---
!pip install -q cloudinary pandas openpyxl requests

In [ ]:
# --- Настройки: заполните и запустите ---

# Cloudinary credentials
CLOUDINARY_CLOUD_NAME = 'ваш_cloud_name'
CLOUDINARY_API_KEY = 'ваш_api_key'
CLOUDINARY_API_SECRET = 'ваш_api_secret'

# Входной Excel (должен быть уже загружен в /content/)
INPUT_FILE = 'royfamily.xlsx'        # или 'dajar-products.xlsx'
BRAND_FOLDER = 'mebrid/royfamily'    # папка в Cloudinary: mebrid/royfamily или mebrid/dajar

# Выходной файл — имя добавит -cloudinary к исходному
OUTPUT_FILE = INPUT_FILE.replace('.xlsx', '-cloudinary.xlsx')

# Лимиты — безопасные значения для бесплатного тарифа
MAX_PHOTOS_PER_PRODUCT = 10   # грузим не больше N фото на товар (экономим кредиты)
SKIP_IF_ALREADY_UPLOADED = True  # пропускать уже залитые (по public_id)

In [ ]:
# --- Инициализация Cloudinary ---
import cloudinary
import cloudinary.uploader
from cloudinary.exceptions import Error as CloudinaryError

cloudinary.config(
    cloud_name = CLOUDINARY_CLOUD_NAME,
    api_key    = CLOUDINARY_API_KEY,
    api_secret = CLOUDINARY_API_SECRET,
    secure     = True,
)

print(f'✓ Cloudinary настроен: {CLOUDINARY_CLOUD_NAME}')

In [ ]:
# --- Читаем Excel и готовим данные ---
import pandas as pd
import re

df = pd.read_excel(INPUT_FILE)
print(f'✓ Загружено {len(df)} товаров из {INPUT_FILE}')
print(f'  Колонки: {list(df.columns)}')
print()
print(df[["ID", "Название"]].head())

In [ ]:
# --- Хелпер: очистить URL от thumbnail-суффиксов ---
def clean_photos(raw: str) -> list[str]:
    """Разбирает текст из колонки "Фото" на список URL, убирает thumbnail-дубли."""
    if not isinstance(raw, str):
        return []
    urls = [u.strip() for u in raw.split('\n') if u.strip()]

    # Убираем дубли вида -300x225 / _thumbnail_1024, если есть полноразмерная версия
    full = set()
    thumbs = {}
    for u in urls:
        base = re.sub(r'-\d+x\d+(?=\.(jpe?g|png|webp))', '', u, flags=re.I)
        base = re.sub(r'_thumbnail_\d+(?=\.(jpe?g|png|webp))', '', base, flags=re.I)
        if base == u:
            full.add(u)
        else:
            thumbs.setdefault(base, u)
    result = list(full)
    for base, thumb in thumbs.items():
        if base not in full:
            result.append(thumb)
    return result[:MAX_PHOTOS_PER_PRODUCT]

# Проверка на одной строке
sample = clean_photos(df.iloc[0]['Фото'])
print(f'✓ У первого товара {len(sample)} уникальных фото (после очистки):')
for u in sample[:3]:
    print(f'  • {u}')

In [ ]:
# --- Основной цикл загрузки ---
from tqdm.auto import tqdm

new_photo_column = []
stats = {'uploaded': 0, 'skipped_existing': 0, 'errors': 0}

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Товары'):
    product_id = row['ID']
    photos = clean_photos(row['Фото'])
    if not photos:
        new_photo_column.append('')
        continue

    uploaded_urls = []
    for i, url in enumerate(photos, start=1):
        public_id = f'{BRAND_FOLDER}/{product_id}/{i:02d}'
        try:
            # Cloudinary умеет грузить напрямую по URL, без скачивания на диск
            result = cloudinary.uploader.upload(
                url,
                public_id = public_id,
                overwrite = not SKIP_IF_ALREADY_UPLOADED,
                resource_type = 'image',
                use_filename = False,
                unique_filename = False,
            )
            uploaded_urls.append(result['secure_url'])
            stats['uploaded'] += 1
        except CloudinaryError as e:
            # Типичные причины: 404 на источнике, rate limit
            msg = str(e)
            if 'already exists' in msg.lower() and SKIP_IF_ALREADY_UPLOADED:
                # Формируем URL по известному формату
                u = f'https://res.cloudinary.com/{CLOUDINARY_CLOUD_NAME}/image/upload/{public_id}'
                uploaded_urls.append(u)
                stats['skipped_existing'] += 1
            else:
                print(f'⚠ [{product_id}] {url} → {msg[:120]}')
                stats['errors'] += 1
        except Exception as e:
            print(f'⚠ [{product_id}] {url} → {type(e).__name__}: {str(e)[:120]}')
            stats['errors'] += 1

    new_photo_column.append('\n'.join(uploaded_urls))

print()
print('=== Готово ===')
print(f'Загружено: {stats["uploaded"]}')
print(f'Уже были в Cloudinary: {stats["skipped_existing"]}')
print(f'Ошибок: {stats["errors"]}')

In [ ]:
# --- Сохраняем новый Excel ---
df_out = df.copy()
df_out['Фото'] = new_photo_column
df_out.to_excel(OUTPUT_FILE, index=False)

print(f'✓ Сохранено: {OUTPUT_FILE}')
print(f'  Размер: {len(df_out)} строк')
print()
print('Первый товар, обновлённая колонка "Фото":')
print(df_out.iloc[0]['Фото'][:500])

## ✅ Готово

- Новый файл лежит в `/content/` — скачайте его через панель слева
- Передайте его разработчику (или положите в `scripts/` своего проекта и запустите `npm run build-data`)
- Для второго файла измените `INPUT_FILE` и `BRAND_FOLDER` в ячейке настроек и запустите снова

## Что дальше — на стороне сайта
После замены URL разработчик должен обновить `next.config.js`:
```js
images: {
  remotePatterns: [
    { protocol: 'https', hostname: 'res.cloudinary.com' },
  ],
  unoptimized: false,  // теперь можно включить оптимизацию
}
```